# Composite Value-for-Money Index for Used Cars

## 1. Theoretical Framework

This notebook will create a Car Value-for-Money Index. 
The aim of the index is to compare used cars using several factors together, instead of judging them by price alone. 
<br>This is important because value-for-money is not based on one single feature. It depends on a combination of cost, age, condition, efficiency, and specification.

This topic was chosen because buying a used car is a realistic problem where several factors need to be considered at the same time. A buyer usually wants a car that is affordable, reliable, practical, and not too expensive to run. 

Autotrader explains that both age and mileage are important when buying a used car, while RAC advises buyers to compare a car’s mileage against its age when checking second-hand cars. The AA also explains that running costs include more than the purchase price, such as fuel and other ownership costs. This supports the idea that a used car should not be judged using only one variable such as price.

### 1.1 Dataset used
The dataset used is the 90,000+ Cars Data From 1970 to 2024 dataset from Kaggle. 
<br>Link to dataset: https://www.kaggle.com/datasets/meruvulikith/90000-cars-data-from-1970-to-2024

It contains used car records with variables such as model, year, price, transmission, mileage, fuel type, tax, MPG, engine size, and manufacturer. 
<br>These columns are suitable for this project because they match the main factors that buyers would normally consider when comparing used cars. For example, 

<br>Price can be used to measure affordability, 
<br>Year can be used to calculate the age of the car, 
<br>Mileage shows how much the car has been used, 
<br>Mpg gives an idea of fuel efficiency, 
<br>Tax gives an indication of running cost. 
<br>The dataset also includes categorical variables such as transmission, fuelType, and Manufacturer, which are useful for grouping and comparing different types of cars.




## 2 Data Cleaning and Preparation

Before carrying out the main analysis, the dataset was checked and cleaned to make sure it was suitable for building the composite index. This stage included checking for missing values, duplicate rows, inconsistent column names, text formatting issues, unrealistic numeric values, and variables that may not be suitable for the final index.

The original dataset contained 97,712 car records and 10 columns. The columns included `model`, `year`, `price`, `transmission`, `mileage`, `fuelType`, `tax`, `mpg`, `engineSize`, and `Manufacturer`.

### 2.1 Missing Values

The dataset was checked for missing values using Pandas. No blank missing values were found in the dataset. This means that no standard missing-value imputation was required at this stage.

Although there were no blank missing values, the data was still inspected for unusual or invalid values. This is important because some values may not appear as missing, but may still be unsuitable for analysis. For example, an engine size of 0 was found in some records. This was investigated further before deciding whether the variable should be used in the final index.

### 2.2 Duplicate Records

The dataset was checked for duplicate rows to avoid counting the same car record more than once. No duplicate rows were found, so no records needed to be removed for duplication.

### 2.3 Column Name Standardisation

Some column names were changed to make them easier to use in Python and to keep the naming style consistent. For example, `fuelType` was renamed to `fuel_type`, `engineSize` was renamed to `engine_size`, and `Manufacturer` was renamed to `manufacturer`.

This made the dataset easier to work with during the analysis and helped avoid errors when referring to column names in the code.

| Original column name | New column name |
|---|---|
| `fuelType` | `fuel_type` |
| `engineSize` | `engine_size` |
| `Manufacturer` | `manufacturer` |

### 2.4 Text Cleaning and Category Fixes

The text-based columns were cleaned to make the values more consistent. This included columns such as `model`, `transmission`, `fuel_type`, and `manufacturer`.

Extra spaces were removed and the text was standardised so that category names were formatted consistently. For example, manufacturer names were changed so that the first letter was capitalised. One spelling issue was found in the manufacturer column, where `hyundi` was corrected to `Hyundai`. The manufacturer `BMW` was also kept in uppercase because it is normally written as an abbreviation.

This cleaning step is useful because inconsistent category names can affect grouping, visualisation, and later analysis.

### 2.5 Feature Engineering: Car Age

A new column called `car_age` was created from the `year` column. This was done using the formula:

`car_age = 2026 - year`

This new variable is easier to interpret than the original year column when comparing cars. For example, a car from 2020 has a car age of 6 years. In the final index, lower car age is considered better because newer cars are usually more desirable and may have less wear.

### 2.6 Checking Unrealistic Values

The numeric columns were checked for unrealistic values. The checks showed that there were no cars with a price less than or equal to 0, no cars with negative mileage, no cars with MPG less than or equal to 0, and no cars with a future year value.

However, 268 cars had an `engine_size` value of 0. This was not automatically removed because some of these records may represent electric vehicles, where traditional engine size in litres is not applicable.

The main checks were:

| Check | Result |
|---|---:|
| Cars with `price <= 0` | 0 |
| Cars with `mileage < 0` | 0 |
| Cars with `mpg <= 0` | 0 |
| Cars with `engine_size <= 0` | 268 |
| Cars with `car_age < 0` | 0 |

### 2.7 Outlier Inspection

After checking for unrealistic values, the numeric variables were also inspected for possible outliers. This was done using summary statistics, percentiles, and boxplots. The main variables checked were `price`, `mileage`, `tax`, `mpg`, and `car_age`.

The table below shows the 1st percentile, 99th percentile, and the number of values outside these limits.

| Variable | 1st Percentile | 99th Percentile | Values Below 1st Percentile | Values Above 99th Percentile |
|---|---:|---:|---:|---:|
| `price` | 3,995.00 | 52,446.92 | 971 | 978 | 
| `mileage` | 11.00 | 95,433.81 | 972 | 978 | 
| `tax` | 0.00 | 265.00 | 0 | 882 | 
| `mpg` | 29.70 | 85.60 | 922 | 906 |
| `car_age` | 6.00 | 16.00 | 1 | 962 |

The boxplots also showed visible outliers for most of these variables, especially `price`, `mileage`, `mpg`, and `car_age`. These outliers were not removed immediately because they may still represent real vehicles. For example, expensive cars may be luxury models, high-mileage cars may be heavily used vehicles, high MPG values may be hybrid or electric cars, and older cars may be classic vehicles.

However, the outliers were noted because they may affect the normalisation stage. If min-max normalisation is applied directly, extreme values can stretch the scale and influence the final Car Value-for-Money Index. Therefore, these outliers will be considered later when choosing and applying the normalisation method.

### 2.8 Variables Excluded from the Final Index

The `engine_size` column was inspected during the cleaning stage. It was originally considered for the specification part of the index, but it was excluded from the final index because engine size is not directly comparable across all fuel types.

For petrol and diesel cars, engine size is usually measured in litres and can describe part of the car’s specification. However, electric vehicles do not have a traditional internal combustion engine measured in the same way. This means that an engine size of 0 may be reasonable for electric cars, but not for petrol or diesel cars. Some electric vehicle records may also have non-zero engine size values, which makes the variable inconsistent for comparison.

To avoid unfair comparisons between electric and non-electric vehicles, `engine_size` removed from the final Car Value-for-Money Index.


### 2.9 Encoding Categorical Variables

Some of the variables in the dataset were categorical, meaning they were stored as text instead of numbers. These included `fuel_type`, `transmission`, `manufacturer`, and `model`. Since later analysis such as correlation analysis and index calculation requires numerical values, the categorical variables had to be handled before they could be used properly.

The `manufacturer` and `model` columns were not encoded for the main index. This is because they contain many different categories and do not have a natural order. For example, it would not be fair to directly assign one manufacturer a higher score than another without using an external reliability or brand ranking source. Instead, these columns were kept for identification, grouping, and visualisation later in the project. The `model` column will help identify cars in the final ranking table, while `manufacturer` can be used to compare average value-for-money scores by brand.

The `fuel_type` column was handled differently because fuel type is related to running cost and efficiency. Instead of manually giving each fuel type a score, a data-based method was used. For each fuel type, the average `mpg` and average `tax` were calculated. Higher MPG was treated as better because it suggests better fuel efficiency, while lower tax was treated as better because it reduces ownership cost.

The MPG and tax values were then normalised so they could be compared on the same scale. A final `fuel_type_score` was created using a weighted combination of the normalised MPG score and the normalised tax score. MPG was given a higher weighting because fuel efficiency is a direct part of running cost, while tax was also included because it contributes to the cost of owning the car.

The resulting fuel type scores were:

| Fuel type | Average MPG | Average Tax | Fuel Type Score |
|---|---:|---:|---:|
| Electric | 297.07 | 22.50 | 1.00 |
| Hybrid | 88.98 | 71.90 | 0.27 |
| Other | 85.76 | 103.64 | 0.17 |
| Diesel | 58.25 | 114.45 | 0.06 |
| Petrol | 50.85 | 127.22 | 0.00 |

Electric cars received the highest fuel type score because they had the highest average MPG and the lowest average tax in the dataset. Petrol cars received the lowest score because they had the lowest average MPG and highest average tax compared with the other fuel types. This does not mean petrol cars are always poor value overall, but only that they scored lowest for this specific fuel efficiency and tax-based part of the index.

The `transmission` column was encoded using one-hot encoding. This method creates a separate column for each transmission type, using a value of 1 if the car belongs to that category and 0 if it does not. One-hot encoding was used because transmission type does not have a clear numerical order. For example, it would be too subjective to say that automatic, manual, semi-auto, or other transmission types are always better or worse.

The transmission categories were encoded into these columns:

| Encoded column | Meaning |
|---|---|
| `transmission_Automatic` | 1 if the car is automatic, otherwise 0 |
| `transmission_Manual` | 1 if the car is manual, otherwise 0 |
| `transmission_Other` | 1 if the car has another/unclear transmission type, otherwise 0 |
| `transmission_Semi-Auto` | 1 if the car is semi-automatic, otherwise 0 |

This approach allowed `fuel_type` and `transmission` to be included in later analysis without forcing all categorical variables into unfair manual rankings.


### 2.10 Final Variables Selected for the Index

After cleaning and inspecting the dataset, the final index was planned around variables that were more consistent and easier to compare across all cars. Some categorical variables were converted into numerical forms so they could be used in later analysis.

| Sub-index | Variables used | Reason |
|---|---|---|
| Affordability Score | `price`, `tax` | Lower price and lower tax make the car cheaper to buy and own. |
| Condition and Age Score | `car_age`, `mileage` | Newer cars and lower-mileage cars are usually more desirable. |
| Efficiency Score | `mpg`, `fuel_type_score` | Better fuel economy can reduce running costs. `fuel_type_score` was calculated using average MPG and tax for each fuel type. |
| Usability Score | `transmission_Automatic`, `transmission_Manual`, `transmission_Other`, `transmission_Semi-Auto` | Transmission was one-hot encoded because it has no clear natural order. |

Overall, the cleaning and preparation process helped make the dataset more consistent and prepared it for the next stage of analysis.

## 3. Multivariate Analysis

## 3. Multivariate Analysis

Multivariate analysis was carried out to understand the relationships between the variables before creating the final Car Value-for-Money Index. This step is important because the index should not be created by combining variables blindly. The variables need to be checked first to see how they relate to each other and whether they provide useful information.

The main variables analysed were `price`, `mileage`, `tax`, `mpg`, `car_age`, and `fuel_type_score`. The `transmission` column was also included in the analysis after being converted into dummy variables using one-hot encoding. The `engine_size` variable was not included because it was found to be unsuitable for direct comparison between electric and non-electric vehicles.

### 3.1 Pearson Correlation with Price

Pearson correlation was used to check the strength and direction of the relationship between `price` and the other selected variables. Price was used as the main comparison variable because it is central to the value-for-money concept.

| Variable compared with `price` | Pearson correlation | Interpretation |
|---|---:|---|
| `mileage` | -0.42 | Moderate negative relationship. Higher mileage is linked with lower price. |
| `tax` | 0.31 | Weak positive relationship. Higher-tax cars tend to have higher prices. |
| `mpg` | -0.30 | Weak negative relationship. Higher MPG cars tend to have slightly lower prices. |
| `car_age` | -0.49 | Moderate negative relationship. Older cars tend to have lower prices. |
| `fuel_type_score` | 0.16 | Very weak positive relationship. Fuel type score has little relationship with price. |
| `transmission_Automatic` | 0.24 | Weak positive relationship. Automatic cars tend to be slightly more expensive. |
| `transmission_Manual` | -0.55 | Moderate negative relationship. Manual cars tend to be cheaper. |
| `transmission_Other` | -0.00 | No clear relationship with price. |
| `transmission_Semi-Auto` | 0.41 | Moderate positive relationship. Semi-auto cars tend to be more expensive. |

The results show that `car_age`, `mileage`, and transmission type have some of the clearest relationships with price. The negative relationship between price and car age is expected, as older cars usually lose value over time. The negative relationship between price and mileage also makes sense because cars with higher mileage have usually been used more and may be less desirable to buyers.

The strongest negative relationship was between `price` and `transmission_Manual`, with a correlation of -0.55. This suggests that manual cars tend to have lower prices in this dataset. On the other hand, `transmission_Semi-Auto` had a moderate positive relationship with price, suggesting that semi-automatic cars tend to be more expensive.

### 3.2 Scatterplot Analysis

<table>
  <tr>
    <td align="center" width="50%">
      <img src="../images/price_mileage.png" width="500">
      <br>
      <em>Figure 1: Price vs Mileage</em>
    </td>
    <td align="center" width="50%">
      <img src="../images/price_age.png" width="500">
      <br>
      <em>Figure 2: Price vs Car Age</em>
    </td>
  </tr>

  <tr>
    <td align="center" width="50%">
      <img src="../images/price_tax.png" width="500">
      <br>
      <em>Figure 3: Price vs Tax</em>
    </td>
    <td align="center" width="50%">
      <img src="../images/price_mpg.png" width="500">
      <br>
      <em>Figure 4: Price vs MPG</em>
    </td>
  </tr>

  <tr>
    <td align="center" width="50%">
      <img src="../images/price_fuel_type.png" width="500">
      <br>
      <em>Figure 5: Price vs Fuel Type Score</em>
    </td>
    <td align="center" width="50%">
      <img src="../images/price_transmission.png" width="500">
      <br>
      <em>Figure 6: Average Price by Transmission Type</em>
    </td>
  </tr>
</table>

Scatterplots were used to visually check the relationships between `price` and the selected numerical variables. A simple regression line was added to each scatterplot to make the general trend easier to see.

The scatterplot of `price` against `mileage` shows a downward trend. This means that as mileage increases, price generally decreases. This supports the Pearson correlation result of -0.42.

The scatterplot of `price` against `car_age` also shows a clear downward trend. Newer cars tend to have higher prices, while older cars tend to be cheaper. This supports the correlation result of -0.49.

The scatterplot of `price` against `tax` shows a weak upward trend. This suggests that higher-tax cars are often more expensive. This may be because more expensive vehicles can have larger engines, higher emissions, or higher specifications.

The scatterplot of `price` against `mpg` shows a weak negative trend. This means that higher-MPG cars tend to be slightly cheaper in this dataset. This may be because many high-MPG cars are smaller economy cars, while larger or higher-performance cars may have lower fuel efficiency.

The `fuel_type_score` plot shows only a weak positive relationship with price. This means fuel type is useful for the efficiency part of the index, but it does not strongly explain price by itself.

### 3.3 Transmission Analysis

The transmission variables were not shown as normal scatterplots because they were encoded as dummy variables with values of 0 or 1. Scatterplots are more useful for continuous variables such as mileage, MPG, tax, and car age.

Instead, a bar chart was used to compare the average price by transmission type. The chart showed that semi-automatic cars had the highest average price, followed by automatic cars. Manual cars had the lowest average price.

This supports the correlation results, where manual transmission had a negative relationship with price, while semi-automatic and automatic transmission had positive relationships with price.

### 3.4 Correlation Matrix Heatmap

A correlation matrix heatmap was also created to show the relationships between all selected variables at the same time. This was useful because the earlier Pearson correlation section mainly focused on how each variable related to `price`, while the heatmap also showed how the other variables related to each other.

<p align="center">
  <img src="../images/correlation_matrix.png" width="750">
</p>

<p align="center"><em>Figure 7: Correlation matrix heatmap for selected variables</em></p>

The heatmap supports the earlier Pearson correlation results. `price` has a negative relationship with `mileage` and `car_age`, meaning higher-mileage and older cars generally have lower prices. This fits the idea that age and usage reduce the value of a used car.

One important relationship shown in the heatmap is between `mileage` and `car_age`, with a correlation of **0.74**. This is a strong positive relationship, meaning older cars generally have higher mileage. This makes sense, but it also means these two variables are partly measuring a similar idea. Because of this, they should not be given too much combined weight in the final index.

The heatmap also shows a moderate negative relationship between `tax` and `mpg`, with a correlation of **-0.45**. This suggests that cars with better fuel efficiency tend to have lower tax. There is also a moderate positive relationship between `mpg` and `fuel_type_score`, with a correlation of **0.46**. This is expected because the fuel type score was partly created using average MPG and tax.

The transmission dummy variables have negative relationships with each other. This is normal because each car can only belong to one transmission category. For example, if a car is manual, it cannot also be automatic or semi-automatic.

### 3.5 Multivariate Analysis Summary

The multivariate analysis shows that the selected variables are useful for understanding used car value. `car_age` and `mileage` are important because they are both negatively related to price, meaning older and more heavily used cars tend to be cheaper. `tax` and `mpg` also provide useful information about running cost and efficiency. `fuel_type_score` has only a weak relationship with price, but it is still useful because it represents fuel efficiency and tax differences between fuel types.

Transmission type also appears to be related to price, but because it is categorical, it is better used for comparison and grouping rather than as a simple continuous score.

Based on this analysis, the following variables will continue to be used when building the index:

| Variable | Decision | Reason |
|---|---|---|
| `price` | Include | Central to affordability and value-for-money. |
| `tax` | Include | Represents part of ownership/running cost. |
| `mileage` | Include | Represents how much the car has been used. |
| `car_age` | Include | Represents the age of the vehicle. |
| `mpg` | Include | Represents fuel efficiency. |
| `fuel_type_score` | Include | Represents fuel type using average MPG and tax. |
| `transmission` | Use for analysis/visualisation | Related to price, but categorical and better used for grouping. |
| `engine_size` | Exclude | Not directly comparable across electric and non-electric vehicles. |

## 4. Linear Regression

Linear regression was used to further examine how the selected variables relate to car price. This was done after the correlation and scatterplot analysis so that the relationships could be checked more formally.

The purpose of this section is not to create the final Car Value-for-Money Index directly. Instead, regression is used as part of the multivariate analysis to understand which variables help explain `price`. This supports the later decisions about variable selection, normalisation, and weighting.

Two types of regression were used:

1. **Simple OLS regression**, where each variable was tested against `price` one at a time.
2. **Multiple linear regression**, where all selected variables were used together to explain `price`.

### 4.1 Simple OLS Regression

Simple OLS regression was first carried out for each selected variable against `price`. This follows the same idea used in the class notes, where one independent variable is tested against a dependent variable to see how much of the variation it explains.

The dependent variable was:

- `price`

The independent variables tested were:

- `mileage`
- `tax`
- `mpg`
- `car_age`
- `fuel_type_score`
- `transmission_Automatic`
- `transmission_Manual`
- `transmission_Other`
- `transmission_Semi-Auto`

The table below summarises the simple OLS regression results.

| Variable | R-squared | Relationship with price | Interpretation |
|---|---:|---|---|
| `mileage` | 0.175 | Negative | Higher mileage is linked with lower price. |
| `tax` | 0.094 | Positive | Higher-tax cars tend to have higher prices. |
| `mpg` | 0.087 | Negative | Higher-MPG cars tend to have lower prices in this dataset. |
| `car_age` | 0.243 | Negative | Older cars tend to have lower prices. |
| `fuel_type_score` | 0.025 | Positive | Fuel type score explains very little of price by itself. |
| `transmission_Automatic` | 0.059 | Positive | Automatic cars tend to be more expensive. |
| `transmission_Manual` | 0.298 | Negative | Manual cars tend to be cheaper. |
| `transmission_Other` | 0.000 | No clear relationship | This category does not explain price well. |
| `transmission_Semi-Auto` | 0.169 | Positive | Semi-automatic cars tend to be more expensive. |

The simple regression results show that the strongest single-variable relationship was between `price` and `transmission_Manual`, with an R-squared value of 0.298. This means that manual transmission alone explained about 29.8% of the variation in price. The coefficient was negative, meaning manual cars tend to be cheaper in this dataset.

The next strongest variable was `car_age`, with an R-squared value of 0.243. This means car age explained about 24.3% of the variation in price. The relationship was negative, which means older cars tend to have lower prices.

`mileage` also had a noticeable relationship with price, with an R-squared value of 0.175. This supports the earlier correlation results, where higher mileage was linked with lower price.

Variables such as `tax`, `mpg`, and `fuel_type_score` had lower R-squared values when tested by themselves. However, they are still useful for the index because they represent running cost and fuel efficiency, not only price.

The `transmission_Other` variable had almost no explanatory power and had a high p-value in the simple OLS output. This suggests that this category does not have a meaningful relationship with price.

### 4.2 Multiple Linear Regression

After testing each variable individually, a multiple linear regression model was created using the selected variables together. This model checks how well the variables explain price as a group.

The dependent variable was:

- `price`

The independent variables were:

- `mileage`
- `tax`
- `mpg`
- `car_age`
- `fuel_type_score`
- `transmission_Automatic`
- `transmission_Manual`
- `transmission_Semi-Auto`

The `transmission_Other` dummy variable was removed from the model to avoid the dummy variable trap. This means it was treated as the reference category for the transmission variables.

The multiple regression model produced an **R-squared value of 0.542**. This means that the selected variables explained approximately **54.2%** of the variation in car price.

This is a useful result because used car prices are affected by many factors, and not all of them are available in the dataset. For example, the dataset does not include service history, vehicle condition, warranty, location, number of previous owners, optional extras, or seller type. Even with these missing factors, the selected variables still explained over half of the variation in price.

### 4.3 Multiple Regression Results

| Variable | Coefficient | P-value | Interpretation |
|---|---:|---:|---|
| `mileage` | -0.0467 | 0.000 | Higher mileage is linked with lower price. |
| `tax` | 9.9182 | 0.000 | Higher tax is linked with higher price. |
| `mpg` | -153.7481 | 0.000 | Higher MPG is linked with lower price in this model. |
| `car_age` | -1485.0465 | 0.000 | Older cars are linked with lower price. |
| `fuel_type_score` | 38360 | 0.000 | Higher fuel type score is linked with higher price. |
| `transmission_Automatic` | 6646.0519 | 0.003 | Automatic cars are more expensive than the reference group. |
| `transmission_Manual` | -26.0404 | 0.991 | Manual transmission is not significant in the multiple model. |
| `transmission_Semi-Auto` | 8613.0749 | 0.000 | Semi-automatic cars are more expensive than the reference group. |

The multiple regression results show that most selected variables were statistically significant. The coefficients for `mileage` and `car_age` were negative, which means that higher-mileage and older cars tend to have lower prices. This supports the earlier scatterplot and Pearson correlation results.

The coefficient for `car_age` was approximately **-1485**. This means that, when the other variables are held constant, each additional year of age is associated with a decrease of about **1,485** in predicted price.

The coefficient for `mileage` was **-0.0467**. This means that higher mileage reduces predicted price. For example, an increase of 10,000 miles would be associated with a decrease of about **467** in predicted price, assuming the other variables stay the same.

The coefficient for `mpg` was negative. This suggests that higher-MPG cars tend to be cheaper in this dataset when the other variables are included. This may be because many high-MPG cars are smaller economy vehicles, while more expensive vehicles may be larger or performance-focused.

The `fuel_type_score` variable had a positive coefficient and was statistically significant. This suggests that cars with a better fuel type score tend to have higher predicted prices when the other variables are controlled for.

For transmission, automatic and semi-automatic cars had positive and statistically significant coefficients compared with the reference category. However, `transmission_Manual` was not statistically significant in the multiple regression model. This is different from the simple OLS result, where manual transmission appeared to be important by itself. This suggests that once other variables such as mileage, car age, MPG, tax, and fuel type are included, manual transmission does not add much extra explanatory power.

### 4.4 Regression Summary

Overall, the regression analysis supports the earlier multivariate analysis. The results show that `car_age`, `mileage`, `mpg`, `tax`, `fuel_type_score`, and some transmission categories are related to car price.

The multiple regression model explained **54.2%** of the variation in price, which suggests that the selected variables are useful for understanding used car value. However, the model also showed a high condition number, which may indicate scaling differences or possible multicollinearity. This is not surprising because the variables are measured on very different scales, such as mileage in thousands, price in currency, and fuel type score between 0 and 1.

Because of this, the regression model is used to support understanding and variable selection, rather than to directly create the final index. The final Car Value-for-Money Index will still be created using normalisation, weighting, and aggregation.

## 5. Principal Component Analysis

Before applying PCA, the selected variables were scaled using `StandardScaler`, because the variables were measured on different scales. For example, `price` and `mileage` have much larger values than `fuel_type_score`, so scaling was needed to prevent larger-number variables from dominating the PCA.

The variables selected for PCA were:

| Variable | Reason for inclusion |
|---|---|
| `price` | Represents affordability and market value |
| `mileage` | Represents how much the car has been used |
| `tax` | Represents part of ownership/running cost |
| `mpg` | Represents fuel efficiency |
| `car_age` | Represents the age of the vehicle |
| `fuel_type_score` | Represents fuel type using average MPG and tax |

The `engine_size` variable was not included because it had already been excluded from the final index due to comparability issues between electric and non-electric vehicles. Transmission dummy variables were also not included in the PCA because they are binary categorical variables and were already analysed separately.

### 5.1 Scree Plot

The scree plot was created to show how much variance was explained by each principal component.

<p align="center">
  <img src="../images/pca_scree_plot.png" width="700">
</p>

<p align="center"><em>Figure 8: PCA scree plot</em></p>

The scree plot shows that PC1 explains 40.7% of the variance, while PC2 explains 24.6%. Together, the first two principal components explain:

`40.7% + 24.6% = 65.3%`

This means that the first two components capture a large amount of the structure in the selected variables. The remaining components explain smaller amounts of variance individually.

### 5.2 PCA Graph

The PCA graph plots the dataset using the first two principal components, PC1 and PC2.

<p align="center">
  <img src="../images/pca_graph.png" width="700">
</p>

<p align="center"><em>Figure 9: PCA graph using PC1 and PC2</em></p>

The PCA graph shows that most cars are grouped around the main cluster. This means many cars have similar patterns across the selected variables. There are also some points further away from the main group. These may represent cars with unusual combinations of values, such as very expensive cars, very old cars, high-mileage cars, or cars with unusual MPG and fuel-type patterns.

This supports the earlier outlier analysis, where variables such as `price`, `mileage`, `mpg`, and `car_age` showed extreme values.

### 5.3 PCA Loading Scores

The loading scores for PC1 were checked to see which original variables contributed most to the first principal component. PC1 was examined because it explained the largest amount of variance in the dataset.

| Variable | PC1 loading score | Interpretation |
|---|---:|---|
| `mileage` | 0.50 | Strong positive contribution |
| `car_age` | 0.50 | Strong positive contribution |
| `price` | -0.45 | Strong negative contribution |
| `tax` | -0.37 | Moderate negative contribution |
| `mpg` | 0.37 | Moderate positive contribution |
| `fuel_type_score` | 0.17 | Weak positive contribution |

The variables with the highest loading scores were `mileage` and `car_age`, both with values of about **0.50**. This means they had the strongest influence on PC1. This suggests that PC1 is mainly affected by how old the car is and how much it has been used.

The variable `price` also had a strong loading score of **-0.45**. The negative value means that `price` moves in the opposite direction to `mileage` and `car_age` on PC1. This makes sense because older cars and higher-mileage cars usually have lower prices.

The variables `tax` and `mpg` had moderate loading scores, while `fuel_type_score` had the lowest loading score. This means that fuel type score had less influence on PC1 compared with mileage, car age, and price.

Overall, PC1 seems to mainly represent the difference between **older, higher-mileage cars** and **higher-priced cars**. This supports the earlier correlation and regression results, where `car_age` and `mileage` were found to be important variables related to price.

### 5.4 PCA Summary

PCA confirms that age, usage, and price are important factors in the dataset. These results support the decision to include `price`, `mileage`, and `car_age` in the final Car Value-for-Money Index. The PCA also supports the decision to group `mileage` and `car_age` together under a Condition and Age Score, because both variables strongly contribute to the same principal component.